# Chapter 52: Vision-Language Models
## Multimedia & Computer Vision Learning Hub — ITERA
**Author:** Aji Pamungkas Tri Nurcahyo  
**Objective:** Memahami arsitektur dual-encoder vision-language (CLIP), memformulasikan fungsi rugi simetris cross-entropy kontrastif, mengevaluasi peran parameter temperatur $\tau$, serta mensimulasikan mekanisme klasifikasi zero-shot berbasis prompt alami dan prompt ensembling.

### 1. Landasan Teori & Formulasi Rugi Simetris CLIP
Diberikan matriks kesamaan kosinus $\mathbf{S} = \mathbf{U} \mathbf{V}^T$ dan logit terskala $\mathbf{L} = \mathbf{S} / \tau$:
$$\mathcal{L}_{v \to t} = -\frac{1}{N} \sum_{i=1}^N \log \left( \frac{\exp(L_{ii})}{\sum_{j=1}^N \exp(L_{ij})} \right)$$
$$\mathcal{L}_{t \to v} = -\frac{1}{N} \sum_{j=1}^N \log \left( \frac{\exp(L_{jj})}{\sum_{i=1}^N \exp(L_{ij})} \right)$$
$$\mathcal{L}_{\text{CLIP}} = \frac{1}{2} (\mathcal{L}_{v \to t} + \mathcal{L}_{t \to v})$$

In [ ]:
# 2. Setup Lingkungan & Dual-Engine Check
import numpy as np
import sys

print(f"Python Version: {sys.version}")
print(f"NumPy Version: {np.__version__}")

HAS_TORCH = False
try:
    import torch
    HAS_TORCH = True
    print(f"PyTorch: {torch.__version__} (Device: {'cuda' if torch.cuda.is_available() else 'cpu'})")
except ImportError:
    print("INFO: PyTorch tidak terpasang di host lokal. Beroperasi dalam Mode Analisis Algoritmik NumPy murni.")

In [ ]:
# 3. Implementasi Rugi Simetris CLIP & Validasi Contoh Analitik Seksi 3
def l2_norm(mat, axis=-1, eps=1e-12):
    norm = np.linalg.norm(mat, axis=axis, keepdims=True)
    return mat / np.maximum(norm, eps)

def compute_clip_loss(U, V, tau=0.20):
    U_norm = l2_norm(U)
    V_norm = l2_norm(V)
    N = len(U)
    
    S = np.dot(U_norm, V_norm.T)
    L = S / tau
    
    # Vision-to-Text (Row Softmax)
    exp_row = np.exp(L - np.max(L, axis=1, keepdims=True))
    p_v2t = exp_row / np.sum(exp_row, axis=1, keepdims=True)
    loss_v2t = -float(np.mean(np.log(np.diag(p_v2t))))
    
    # Text-to-Vision (Column Softmax)
    exp_col = np.exp(L - np.max(L, axis=0, keepdims=True))
    p_t2v = exp_col / np.sum(exp_col, axis=0, keepdims=True)
    loss_t2v = -float(np.mean(np.log(np.diag(p_t2v))))
    
    loss_clip = 0.5 * (loss_v2t + loss_t2v)
    return loss_clip, loss_v2t, loss_t2v, S, L, p_v2t, p_t2v

# Pasangan Uji N=2, D=2 sesuai Seksi 3
U = np.array([[1.0, 0.0], [0.0, 1.0]], dtype=np.float64)
V = np.array([[0.8, 0.6], [-0.6, 0.8]], dtype=np.float64)
tau = 0.20

loss, lv, lt, S, L, pv, pt = compute_clip_loss(U, V, tau=tau)
print(f"Matriks Kesamaan S:\n{S}")
print(f"Matriks Logit L:\n{L}")
print(f"Loss v2t: {lv:.7f} | Loss t2v: {lt:.7f}")
print(f"Total CLIP Loss: {loss:.7f}")

# Assertions
np.testing.assert_allclose(S, [[0.8, -0.6], [0.6, 0.8]], atol=1e-5)
np.testing.assert_allclose(L, [[4.0, -3.0], [3.0, 4.0]], atol=1e-5)
np.testing.assert_allclose(loss, 0.1570866, atol=1e-5)
print("VALIDASI BERHASIL: Loss simetris CLIP cocok presisi sesuai perhitungan manual!")

In [ ]:
# 4. Eksperimen Klasifikasi Zero-Shot & Prompt Ensembling
classes = ["kucing", "anjing"]
# U[0] adalah Citra Kucing
sims = np.dot(V, U[0])
logits = sims / tau
probs = np.exp(logits - np.max(logits)) / np.sum(np.exp(logits - np.max(logits)))

pred_idx = np.argmax(probs)
print(f"Prediksi Zero-Shot: '{classes[pred_idx]}' dengan keyakinan {probs[pred_idx]*100:.2f}%")
assert pred_idx == 0, "Prediksi harus menghasilkan kelas kucing!"

# Simulasi Prompt Ensembling: Menggabungkan dua variasi template teks
# Template A: 'a photo of a {c}' -> V
# Template B: 'a close-up photo of the {c}' -> V_b (sedikit berbeda)
V_b = np.array([[0.82, 0.58], [-0.58, 0.82]], dtype=np.float64)
V_ens = l2_norm(0.5 * (V + V_b))
sims_ens = np.dot(V_ens, U[0])
print(f"Kesamaan Kosinus setelah Prompt Ensembling: {sims_ens[0]:.4f}")
print("VERIFIKASI ZERO-SHOT & ENSEMBLING SELESAI: Lulus deterministik.")

In [ ]:
# 5. Spesifikasi Eksekusi Pretrained OpenCLIP (Untuk Google Colab / GPU)
if HAS_TORCH:
    print("Mode PyTorch / GPU Aktif.")
    U_t = torch.tensor(U, dtype=torch.float32)
    V_t = torch.tensor(V, dtype=torch.float32)
    L_t = torch.matmul(U_t, V_t.T) / tau
    print("PyTorch Logits Shape:", L_t.shape)
else:
    print("PANDUAN EKSEKUSI MODEL VISION-LANGUAGE PRETRAINED DI GOOGLE COLAB:")
    print("1. Pasang paket open_clip_torch:")
    print("   !pip install open_clip_torch torch torchvision pillow")
    print("2. Muat model ViT-B-32:")
    print("   import open_clip, torch, PIL.Image")
    print("   model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')")
    print("   tokenizer = open_clip.get_tokenizer('ViT-B-32')")
    print("3. Jalankan inferensi zero-shot:")
    print("   image = preprocess(PIL.Image.open('test.jpg')).unsqueeze(0)")
    print("   text = tokenizer(['a photo of a cat', 'a photo of a dog', 'a photo of a car'])")
    print("   with torch.no_grad():")
    print("       image_features = model.encode_image(image)")
    print("       text_features = model.encode_text(text)")
    print("       image_features /= image_features.norm(dim=-1, keepdim=True)")
    print("       text_features /= text_features.norm(dim=-1, keepdim=True)")
    print("       probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)")
    print("   print('Zero-shot prediction probabilities:', probs)")

### 6. Rangkuman & Pembahasan Akademik
1. **Representasi Bersama**: Arsitektur dual-encoder memetakan citra dan teks ke manifold unit hypersphere bersama untuk inferensi zero-shot instan.
2. **Loss Simetris**: Optimasi simultan citra-ke-teks dan teks-ke-citra menjaga kesetimbangan gradien antara kedua modalitas.
3. **Tantangan Praktis**: Penguasaan prompt ensembling dan pemahaman keterbatasan model terhadap halusinasi visual serta relasi spasial kompleks adalah kompetensi esensial insinyur VLM.